# WikiStreams lab
Read Wikipedia edits from Kafka and compare the windows. All times are UTC.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/jobs')
from wikistream_consumer import create_spark, read_events, aggregations, start_queries
spark = create_spark()

In [ ]:
events = read_events(spark)
events.printSchema()
frames = aggregations(events)

## Bot edits
The demo counts bot and human edits in one-hour windows.

In [ ]:
frames['bot_edits_1h'].explain()

## Tumbling windows
A one-minute window has no overlap. Each edit belongs to one window. We count edits and measure the size change in bytes.

In [ ]:
frames['tumbling_edits_1m'].explain()

## Overlapping windows
The window lasts two minutes and starts every 30 seconds. Each edit belongs to four windows. Counts from these windows should not be added together.

In [ ]:
frames['sliding_edits_2m'].explain()

## Large edits
This query keeps edits with a size change of at least 500 bytes and groups them by page. The size is the absolute difference between the old and new page lengths.

In [ ]:
frames['large_page_edits'].explain()

## Start the queries
Watch the output with `docker logs -f wikistreams_notebook`. Update mode prints changed rows. A two-minute watermark limits old state and may drop very late events.

In [ ]:
for query in spark.streams.active:
    query.stop()
queries = start_queries(frames)
[(query.name, query.isActive) for query in queries]

In [ ]:
for query in queries:
    query.processAllAvailable()
    print(query.name, query.isActive, query.exception())
    assert query.exception() is None


## Producer filters
The default is French Wikipedia edits. Use `--server en.wikipedia.org` for English, `--title France` for one exact page title, or `--bot no` for human edits. These filters run before sending events to Kafka.

To stop later, run `for query in queries: query.stop()` in a new cell.